In [1]:
import os
import tempfile
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from tqdm.auto import tqdm

from datetime import datetime

from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
)

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer


from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import VarianceThreshold

from sklearn.utils.class_weight import compute_sample_weight
import inspect


import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tempfile import NamedTemporaryFile

In [2]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        current_year: int | None = None,
        vintage_age_years: int = 25,
        high_mileage_quantile: float = 0.75,
    ) -> None:
        self.current_year = current_year
        self.vintage_age_years = vintage_age_years
        self.high_mileage_quantile = high_mileage_quantile

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        current_year = self._current_year()

        if "yearOfRegistration" in X_df.columns:
            vehicle_age = current_year - pd.to_numeric(
                X_df["yearOfRegistration"], errors="coerce"
            )
            vehicle_age = vehicle_age.clip(lower=0)
            km_per_year = pd.to_numeric(X_df.get("kilometer"), errors="coerce") / (
                vehicle_age + 1
            )
            self.high_mileage_threshold_ = float(
                km_per_year.quantile(self.high_mileage_quantile)
            )
        else:
            self.high_mileage_threshold_ = 0.0

        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        current_year = self._current_year()

        year = pd.to_numeric(X_df.get("yearOfRegistration"), errors="coerce")
        power = pd.to_numeric(X_df.get("power"), errors="coerce")
        km = pd.to_numeric(X_df.get("kilometer"), errors="coerce")

        vehicle_age = (current_year - year).clip(lower=0)
        X_df["vehicleAge"] = vehicle_age

        X_df["kmPerYear"] = km / (vehicle_age + 1)
        X_df["km_power_ratio"] = km / (power + 1)

        X_df["log_power"] = np.log1p(power.clip(lower=0))
        X_df["power_per_age"] = power / (vehicle_age + 1)

        X_df["log_kilometer"] = np.log1p(km.clip(lower=0))

        X_df["is_vintage"] = (vehicle_age > self.vintage_age_years).astype(int)
        thr = getattr(self, "high_mileage_threshold_", 0.0)
        X_df["high_mileage_flag"] = (X_df["kmPerYear"] > thr).astype(int)

        X_df = X_df.drop(columns=["yearOfRegistration"], errors="ignore")
        return X_df

    def _current_year(self) -> int:
        if self.current_year is not None:
            return int(self.current_year)
        return datetime.now().year

In [3]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols: list[str]) -> None:
        self.cols = cols

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        self.freq_maps_ = {}
        for col in list(self.cols):
            if col in X_df.columns:
                self.freq_maps_[col] = X_df[col].value_counts(normalize=True)
            else:
                self.freq_maps_[col] = pd.Series(dtype=float)
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        for col in list(self.cols):
            freq = self.freq_maps_.get(col)
            X_df[f"{col}_freq"] = (
                X_df[col].map(freq).fillna(0.0) if col in X_df.columns else 0.0
            )
        return X_df

In [4]:
def build_preprocess() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                'num',
                Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='median')) ,
                    ('scaler', StandardScaler()),
                ]),
                make_column_selector(dtype_include=np.number),
            ),
            (
                'cat',
                Pipeline(steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')),
                ]),
                make_column_selector(dtype_include=object),
            ),
        ],
        remainder='drop',
    )

In [5]:
def business_metrics(y_true, y_pred) -> dict[str, float]:
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    luxury_recall = float(report.get("2", {}).get("recall", 0.0))

    y_t = pd.Series(y_true).reset_index(drop=True)
    y_p = pd.Series(y_pred).reset_index(drop=True)

    severe = ((y_t == 0) & (y_p == 2)) | ((y_t == 2) & (y_p == 0))
    severe_rate = float(severe.mean())

    return {
        'luxury_recall': luxury_recall,
        'severe_misclassification_rate': severe_rate,
    }

In [6]:
model_configs = {
    "random_forest": {
        "estimator": RandomForestClassifier(random_state=42, n_jobs=1),
        "param_distributions": {
            "model__n_estimators": [100, 200, 300, 500],
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2"],
            "model__bootstrap": [True, False],
        },
        "n_iter": 3,
    },
    "log_reg": {
        "estimator": LogisticRegression(max_iter=2500, solver="saga", random_state=42),
        "param_distributions": {"model__C": np.logspace(-2, 1, 6)},
        "n_iter": 6,
    },
    "linear_svc": {
        "estimator": LinearSVC(random_state=42),
        "param_distributions": {
            "model__C": np.logspace(-3, 2, 6),
            "model__loss": ["hinge", "squared_hinge"],
        },
        "n_iter": 8,
    },
    "extra_trees": {
        "estimator": ExtraTreesClassifier(random_state=42, n_jobs=1),
        "param_distributions": {
            "model__n_estimators": [100, 200, 300, 500],
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2"],
            "model__bootstrap": [False, True],
        },
        "n_iter": 8,
    },
    "decision_tree": {
        "estimator": DecisionTreeClassifier(random_state=42),
        "param_distributions": {
            "model__max_depth": [None, 5, 10, 20, 30],
            "model__min_samples_split": [2, 5, 10, 20],
            "model__min_samples_leaf": [1, 2, 4, 8],
            "model__criterion": ["gini", "entropy"],
            "model__max_features": [None, "sqrt", "log2"],
        },
        "n_iter": 8,
    },
    "xgboost": {
        "estimator": XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            random_state=42,
            n_jobs=1,
            tree_method="hist",
        ),
        "param_distributions": {
            "model__n_estimators": [200, 300, 500],
            "model__max_depth": [4, 6, 8, 10],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__subsample": [0.7, 0.8, 1.0],
            "model__colsample_bytree": [0.7, 0.8, 1.0],
            "model__gamma": [0, 1, 5],
        },
        "n_iter": 8,
    },
    "baseline_dummy": {
        "estimator": DummyClassifier(),
        "param_distributions": {
            "model__strategy": ["most_frequent", "prior", "stratified"]
        },
        "n_iter": 3,
    }
}

In [7]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("used_car_price_tier_classification")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

best_estimators = {}
best_scores = {}

mlflow.end_run()

In [8]:
def build_pipeline(
    estimator,
    preprocess: ColumnTransformer,
    strategy: str,
    current_year: int | None,
    use_feature_selection: bool,
):
    steps = [
        ("fe", FeatureEngineer(current_year=current_year)),
        ("freq", FrequencyEncoder(["brand", "model"])),
        (
            "drop_cols",
            FunctionTransformer(
                lambda X: X.drop(columns=["brand", "model"], errors="ignore"),
                validate=False,
            ),
        ),
        ("preprocess", preprocess),
    ]

    if use_feature_selection:
        selector = SelectFromModel(
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
            ),
            threshold="median",
        )
        steps.append(("feature_selection", selector))

    if strategy == "smote":
        steps.append(("resample", SMOTE(random_state=42)))
    elif strategy == "undersample":
        steps.append(("resample", RandomUnderSampler(random_state=42)))
    elif strategy != "none":
        raise ValueError("strategy must be one of: none, smote, undersample")

    steps.append(("model", estimator))
    return Pipeline(steps)

In [9]:
def run_experiments(
    model_configs: dict[str, dict],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    strategy: str,
    preprocess: ColumnTransformer,
    cv: StratifiedKFold,
    current_year: int | None,
    use_feature_selection: bool,
    reports_dir: Path,
    model_dir: Path,
):
    results: list[dict] = []
    best_estimators: dict[str, Pipeline] = {}

    for model_name, cfg in model_configs.items():
        pipe = build_pipeline(
            cfg["estimator"],
            preprocess=preprocess,
            strategy=strategy,
            current_year=current_year,
            use_feature_selection=use_feature_selection,
        )

        search = RandomizedSearchCV(
            pipe,
            cfg["param_distributions"],
            n_iter=cfg["n_iter"],
            scoring={
                "macro_f1": "f1_macro",
                "balanced_accuracy": "balanced_accuracy",
            },
            refit="macro_f1",
            cv=cv,
            n_jobs=1,
            random_state=42,
        )

        fs_tag = "fs" if use_feature_selection else "no_fs"
        with mlflow.start_run(run_name=f"{model_name}_{strategy}_{fs_tag}"):
            search.fit(X_train, y_train)

            best_model = search.best_estimator_
            best_idx = int(search.best_index_)
            cv_f1 = float(search.cv_results_["mean_test_macro_f1"][best_idx])
            cv_bal = float(search.cv_results_["mean_test_balanced_accuracy"][best_idx])

            y_pred_train = best_model.predict(X_train)
            train_macro_f1 = float(f1_score(y_train, y_pred_train, average="macro"))
            train_bal_acc = float(balanced_accuracy_score(y_train, y_pred_train))
            train_biz = business_metrics(y_train, y_pred_train)

            y_pred = best_model.predict(X_test)
            test_macro_f1 = float(f1_score(y_test, y_pred, average="macro"))
            test_bal_acc = float(balanced_accuracy_score(y_test, y_pred))
            biz = business_metrics(y_test, y_pred)

            cm = confusion_matrix(y_test, y_pred)
            plt.figure(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt="d")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.title(f"{model_name} - {strategy}")
            with NamedTemporaryFile(suffix=".png", delete=False) as tmp:
                plt.savefig(tmp.name, bbox_inches="tight")
                mlflow.log_artifact(tmp.name, "confusion_matrix")
            plt.close()

            report_text = classification_report(y_test, y_pred, zero_division=0)
            report_path = (
                reports_dir / f"classification_report_{model_name}_{strategy}.txt"
            )
            report_path.write_text(report_text, encoding="utf-8")
            mlflow.log_artifact(str(report_path), "reports")

            train_report_text = classification_report(
                y_train, y_pred_train, zero_division=0
            )
            train_report_path = (
                reports_dir / f"classification_report_train_{model_name}_{strategy}.txt"
            )
            train_report_path.write_text(train_report_text, encoding="utf-8")
            mlflow.log_artifact(str(train_report_path), "reports")

            mlflow.log_param("model", model_name)
            mlflow.log_param("strategy", strategy)
            mlflow.log_param(
                "current_year", current_year if current_year is not None else "system"
            )
            mlflow.log_param("feature_selection", bool(use_feature_selection))
            for k, v in search.best_params_.items():
                mlflow.log_param(k, v)
            mlflow.log_param("cv_folds", cv.get_n_splits())

            mlflow.log_metric("cv_macro_f1", cv_f1)
            mlflow.log_metric("cv_balanced_accuracy", cv_bal)
            mlflow.log_metric("train_macro_f1", train_macro_f1)
            mlflow.log_metric("train_balanced_accuracy", train_bal_acc)
            mlflow.log_metric("train_luxury_recall", train_biz["luxury_recall"])
            mlflow.log_metric(
                "train_severe_misclassification_rate",
                train_biz["severe_misclassification_rate"],
            )
            mlflow.log_metric("test_macro_f1", test_macro_f1)
            mlflow.log_metric("test_balanced_accuracy", test_bal_acc)
            mlflow.log_metric("test_luxury_recall", biz["luxury_recall"])
            mlflow.log_metric(
                "test_severe_misclassification_rate",
                biz["severe_misclassification_rate"],
            )

            mlflow.sklearn.log_model(best_model, "model")

            model_out = model_dir / f"{model_name}_{strategy}.pkl"
            mlflow.sklearn.save_model(best_model, path=str(model_out))

            results.append(
                {
                    "model": model_name,
                    "strategy": strategy,
                    "cv_macro_f1": cv_f1,
                    "cv_balanced_accuracy": cv_bal,
                    "train_macro_f1": train_macro_f1,
                    "train_balanced_accuracy": train_bal_acc,
                    "train_luxury_recall": train_biz["luxury_recall"],
                    "train_severe_misclassification_rate": train_biz[
                        "severe_misclassification_rate"
                    ],
                    "test_macro_f1": test_macro_f1,
                    "test_balanced_accuracy": test_bal_acc,
                    "test_luxury_recall": biz["luxury_recall"],
                    "test_severe_misclassification_rate": biz[
                        "severe_misclassification_rate"
                    ],
                }
            )
            best_estimators[f"{model_name}_{strategy}"] = best_model

    return pd.DataFrame(results), best_estimators

In [10]:
df = pd.read_csv("../data/processed/clean_data.csv")
    
def price_tier(price):
    if price < 5000:
        return "budget"
    elif price < 15000:
        return "mid-range"
    else:
        return "luxury"


df["price_tier"] = df["price"].apply(price_tier)

if "price_tier" not in df.columns:
    raise ValueError("Expected 'price_tier' column in validated dataset")

df = df.copy()
df["price_tier"] = (
    df["price_tier"]
    .astype(str)
    .str.lower()
    .map({"budget": 0, "mid-range": 1, "luxury": 2})
)
df = df.dropna(subset=["price_tier"]).copy()

X = df.drop(columns=["price_tier", "price"], errors="ignore")
y = df["price_tier"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

In [ ]:
DEFAULT_REPORTS_DIR = Path('reports/results')
DEFAULT_MODEL_DIR = Path('models')
DEFAULT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DEFAULT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

preprocess = build_preprocess()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


results_smote, best_estimators_smote = run_experiments(
    model_configs=model_configs,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    strategy="smote",
    preprocess=preprocess,
    cv=cv,
    current_year=datetime.now().year,
    use_feature_selection=True,
    reports_dir=DEFAULT_REPORTS_DIR,
    model_dir=DEFAULT_MODEL_DIR,
)

In [ ]:
results_unsersample, best_estimators_undersample = run_experiments(
    model_configs=model_configs,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    strategy="undersample",
    preprocess=preprocess,
    cv=cv,
    current_year=datetime.now().year,
    use_feature_selection=True,
    reports_dir=DEFAULT_REPORTS_DIR,
    model_dir=DEFAULT_MODEL_DIR,
)


In [ ]:
results_none, best_estimators_none = run_experiments(
    model_configs=model_configs,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    strategy="none",
    preprocess=preprocess,
    cv=cv,
    current_year=datetime.now().year,
    use_feature_selection=True,
    reports_dir=DEFAULT_REPORTS_DIR,
    model_dir=DEFAULT_MODEL_DIR,
)

In [ ]:
results_smote, best_estimators_smote = run_experiments(
    model_configs=model_configs,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    strategy="smote",
    preprocess=preprocess,
    cv=cv,
    current_year=datetime.now().year,
    use_feature_selection=False,
    reports_dir=DEFAULT_REPORTS_DIR,
    model_dir=DEFAULT_MODEL_DIR,
)
results_unsersample, best_estimators_undersample = run_experiments(
    model_configs=model_configs,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    strategy="undersample",
    preprocess=preprocess,
    cv=cv,
    current_year=datetime.now().year,
    use_feature_selection=False,
    reports_dir=DEFAULT_REPORTS_DIR,
    model_dir=DEFAULT_MODEL_DIR,
)
results_none, best_estimators_none = run_experiments(
    model_configs=model_configs,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    strategy="none",
    preprocess=preprocess,
    cv=cv,
    current_year=datetime.now().year,
    use_feature_selection=False,
    reports_dir=DEFAULT_REPORTS_DIR,
    model_dir=DEFAULT_MODEL_DIR,
)